In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path("/Users/yongyili/urban-emotional-intelligence")
DATA_DIR = PROJECT_ROOT / "data"

print("Project root:", PROJECT_ROOT)
print("Data folder:", DATA_DIR)

Project root: /Users/yongyili/urban-emotional-intelligence
Data folder: /Users/yongyili/urban-emotional-intelligence/data


In [2]:
features_path = DATA_DIR / "vlm_features.parquet"

if not features_path.exists():
    raise FileNotFoundError("data/vlm_features.parquet not found. Please finish Stage 2 first.")

df = pd.read_parquet(features_path)

print("Loaded VLM features:", df.shape)
df.head()

Loaded VLM features: (655, 20)


,image_id,filepath,borough,road_type,sample_latitude,sample_longitude,latitude,longitude,compass_angle,captured_at,source,scene_description,sky_fraction,greenery_present,greenery_density,building_height_ratio,traffic_intensity,pedestrian_activity,surface_condition,lighting
0,2567577653535954,images/Kingston_upon_Thames/2567577653535954.jpg,Kingston upon Thames,unknown,51.351090,-0.305450,51.351651,-0.312890,13.042664,1499686461379,Mapillary,A road intersection with traffic lights and a ...,0.6,True,medium,low,light,none,fair,bright
1,227868632440032,images/Kingston_upon_Thames/227868632440032.jpg,Kingston upon Thames,unknown,51.401414,-0.262992,51.403125,-0.261115,277.307770,1414503548000,Mapillary,A tree-lined paved path with a grassy verge an...,0.1,True,high,low,none,low,good,bright
2,1101120203722336,images/Kingston_upon_Thames/1101120203722336.jpg,Kingston upon Thames,unknown,51.348385,-0.328315,51.351438,-0.330623,58.696777,1545651013361,Mapillary,"A multi-lane highway with cars driving on it, ...",0.6,True,low,low,moderate,none,good,normal
3,220316326204880,images/Kingston_upon_Thames/220316326204880.jpg,Kingston upon Thames,unknown,51.386792,-0.270959,51.387736,-0.271491,44.678802,1612180724046,Mapillary,A multi-story red brick building with white tr...,0.5,True,low,medium,moderate,low,fair,dim
4,160502959355506,images/Kingston_upon_Thames/160502959355506.jpg,Kingston upon Thames,unknown,51.391711,-0.310473,51.392385,-0.310321,213.639465,1606489867752,Mapillary,"A street scene with buildings on both sides, a...",0.5,True,low,medium,moderate,none,fair,normal


In [3]:
required_cols = [
    "image_id",
    "filepath",
    "borough",
    "road_type",
    "latitude",
    "longitude",
    "scene_description",
    "sky_fraction",
    "greenery_present",
    "greenery_density",
    "building_height_ratio",
    "traffic_intensity",
    "pedestrian_activity",
    "surface_condition",
    "lighting"
]

missing = [c for c in required_cols if c not in df.columns]

if missing:
    raise ValueError(f"Missing required columns: {missing}")

print("All required Stage 3 input columns are present.")

All required Stage 3 input columns are present.


In [4]:
df[[
    "sky_fraction",
    "greenery_present",
    "greenery_density",
    "building_height_ratio",
    "traffic_intensity",
    "pedestrian_activity",
    "surface_condition",
    "lighting"
]].head()

,sky_fraction,greenery_present,greenery_density,building_height_ratio,traffic_intensity,pedestrian_activity,surface_condition,lighting
0,0.6,True,medium,low,light,none,fair,bright
1,0.1,True,high,low,none,low,good,bright
2,0.6,True,low,low,moderate,none,good,normal
3,0.5,True,low,medium,moderate,low,fair,dim
4,0.5,True,low,medium,moderate,none,fair,normal


In [12]:
# Make a copy for Stage 3 processing
scores_df = df.copy()

# Standardise text feature columns
text_feature_cols = [
    "greenery_density",
    "building_height_ratio",
    "traffic_intensity",
    "pedestrian_activity",
    "surface_condition",
    "lighting"
]

for col in text_feature_cols:
    scores_df[col] = scores_df[col].astype(str).str.strip().str.lower()

# Ensure sky_fraction is numeric and clipped to [0, 1]
scores_df["sky_fraction"] = pd.to_numeric(scores_df["sky_fraction"], errors="coerce")
scores_df["sky_fraction"] = scores_df["sky_fraction"].clip(0, 1)

# Ensure greenery_present is boolean
if scores_df["greenery_present"].dtype != bool:
    scores_df["greenery_present"] = scores_df["greenery_present"].astype(str).str.lower().map({
        "true": True,
        "false": False,
        "1": True,
        "0": False
    })

print(scores_df[[
    "sky_fraction",
    "greenery_present",
    "greenery_density",
    "building_height_ratio",
    "traffic_intensity",
    "pedestrian_activity",
    "surface_condition",
    "lighting"
]].isna().sum())

sky_fraction             563
greenery_present           0
greenery_density           0
building_height_ratio      0
traffic_intensity          0
pedestrian_activity        0
surface_condition          0
lighting                   0
dtype: int64


In [6]:
# check original sky_fraction value
df["sky_fraction"].head(30)

0     0.6
1     0.1
2     0.6
3     0.5
4     0.5
5     0.2
6     0.7
7     0.7
8     0.0
9     0.5
10    0.1
11    0.7
12    0.6
13    0.3
14    0.6
15    0.6
16    0.6
17    0.4
18    0.6
19    0.5
20    0.7
21    0.7
22    0.3
23    0.2
24    0.7
25    0.7
26    0.6
27    0.0
28    0.1
29    0.1
Name: sky_fraction, dtype: float64

In [7]:
df["sky_fraction"].value_counts(dropna=False).head(30)

sky_fraction
NaN    563
0.7     24
0.6     21
0.5     13
0.3     13
0.1      7
0.2      6
0.0      4
0.4      4
Name: count, dtype: int64

In [8]:
df["sky_fraction"].apply(type).value_counts()

sky_fraction
<class 'float'>    655
Name: count, dtype: int64

In [10]:
#Use a safer parser for sky_fraction
import re
import numpy as np

def parse_sky_fraction(value):
    """
    Convert VLM sky_fraction outputs into a float between 0 and 1.
    Handles numbers, strings, percentages, and rough categories.
    """
    if pd.isna(value):
        return np.nan

    # Already numeric
    if isinstance(value, (int, float, np.integer, np.floating)):
        v = float(value)
        if v > 1 and v <= 100:
            v = v / 100
        return max(0.0, min(1.0, v))

    text = str(value).strip().lower()

    # Category fallback
    category_map = {
        "none": 0.0,
        "no sky": 0.0,
        "not visible": 0.0,
        "low": 0.2,
        "small": 0.2,
        "limited": 0.2,
        "medium": 0.5,
        "moderate": 0.5,
        "some": 0.4,
        "high": 0.8,
        "large": 0.8,
        "open": 0.8,
        "mostly sky": 0.9
    }

    if text in category_map:
        return category_map[text]

    # Handle percentages like "30%"
    percent_match = re.search(r"(\d+\.?\d*)\s*%", text)
    if percent_match:
        v = float(percent_match.group(1)) / 100
        return max(0.0, min(1.0, v))

    # Handle strings containing a number like "approximately 0.3"
    number_match = re.search(r"(\d+\.?\d*)", text)
    if number_match:
        v = float(number_match.group(1))
        if v > 1 and v <= 100:
            v = v / 100
        return max(0.0, min(1.0, v))

    return np.nan

In [11]:
scores_df["sky_fraction_raw"] = scores_df["sky_fraction"]

scores_df["sky_fraction"] = scores_df["sky_fraction_raw"].apply(parse_sky_fraction)

print("Missing sky_fraction after parsing:", scores_df["sky_fraction"].isna().sum())

scores_df[["sky_fraction_raw", "sky_fraction"]].head(20)

Missing sky_fraction after parsing: 563


,sky_fraction_raw,sky_fraction
0,0.6,0.6
1,0.1,0.1
2,0.6,0.6
3,0.5,0.5
4,0.5,0.5
5,0.2,0.2
6,0.7,0.7
7,0.7,0.7
8,0.0,0.0
9,0.5,0.5


In [13]:
#fill them based on building_height_ratio, because high buildings usually imply lower sky visibility.
def fallback_sky_from_height(row):
    if pd.notna(row["sky_fraction"]):
        return row["sky_fraction"]

    height = str(row["building_height_ratio"]).lower()

    if height == "high":
        return 0.2
    elif height == "medium":
        return 0.4
    elif height == "low":
        return 0.6
    else:
        return 0.4

scores_df["sky_fraction"] = scores_df.apply(fallback_sky_from_height, axis=1)

print("Missing sky_fraction after fallback:", scores_df["sky_fraction"].isna().sum())
scores_df["sky_fraction"].describe()

Missing sky_fraction after fallback: 0


count    655.000000
mean       0.484733
std        0.140919
min        0.000000
25%        0.400000
50%        0.600000
75%        0.600000
max        0.700000
Name: sky_fraction, dtype: float64

In [14]:
scores_df[[
    "sky_fraction",
    "greenery_present",
    "greenery_density",
    "building_height_ratio",
    "traffic_intensity",
    "pedestrian_activity",
    "surface_condition",
    "lighting"
]].isna().sum()

sky_fraction             0
greenery_present         0
greenery_density         0
building_height_ratio    0
traffic_intensity        0
pedestrian_activity      0
surface_condition        0
lighting                 0
dtype: int64

In [16]:
#encode categorical features numerically
# Ordinal encodings
low_med_high = {
    "low": 1,
    "medium": 2,
    "high": 3
}

none_low_high = {
    "none": 0,
    "low": 1,
    "medium": 2,
    "high": 3
}

traffic_map = {
    "none": 0,
    "light": 1,
    "moderate": 2,
    "heavy": 3
}

condition_map = {
    "poor": 0,
    "fair": 1,
    "good": 2,
    "excellent": 3
}

lighting_map = {
    "dark": 0,
    "dim": 1,
    "normal": 2,
    "bright": 3
}

scores_df["height_enc"] = scores_df["building_height_ratio"].map(low_med_high).fillna(2)
scores_df["traffic_enc"] = scores_df["traffic_intensity"].map(traffic_map).fillna(1)
scores_df["pedestrian_enc"] = scores_df["pedestrian_activity"].map(none_low_high).fillna(1)
scores_df["greenery_enc"] = scores_df["greenery_density"].map(none_low_high).fillna(1)
scores_df["surface_enc"] = scores_df["surface_condition"].map(condition_map).fillna(1)
scores_df["light_enc"] = scores_df["lighting"].map(lighting_map).fillna(2)
scores_df["green_bin"] = scores_df["greenery_present"].astype(int)

scores_df[[
    "height_enc",
    "traffic_enc",
    "pedestrian_enc",
    "greenery_enc",
    "surface_enc",
    "light_enc",
    "green_bin"
]].describe()

,height_enc,traffic_enc,pedestrian_enc,greenery_enc,surface_enc,light_enc,green_bin
count,655.000000,655.000000,655.000000,655.000000,655.000000,655.000000,655.000000
mean,1.520611,1.064122,0.326718,1.532824,1.345038,2.129771,0.993893
std,0.610148,0.802621,0.533413,0.877030,0.541859,0.760220,0.077967
min,1.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000
25%,1.000000,0.000000,0.000000,1.000000,1.000000,2.000000,1.000000
50%,1.000000,1.000000,0.000000,2.000000,1.000000,2.000000,1.000000
75%,2.000000,2.000000,1.000000,2.000000,2.000000,3.000000,1.000000
max,3.000000,3.000000,2.000000,3.000000,3.000000,3.000000,1.000000


In [17]:
encoded_cols = [
    "height_enc",
    "traffic_enc",
    "pedestrian_enc",
    "greenery_enc",
    "surface_enc",
    "light_enc",
    "green_bin"
]

for col in encoded_cols:
    print("\n" + col)
    print(scores_df[col].value_counts(dropna=False).sort_index())


height_enc
height_enc
1    354
2    261
3     40
Name: count, dtype: int64

traffic_enc
traffic_enc
0    188
1    240
2    224
3      3
Name: count, dtype: int64

pedestrian_enc
pedestrian_enc
0    462
1    172
2     21
Name: count, dtype: int64

greenery_enc
greenery_enc
0     78
1    242
2    243
3     92
Name: count, dtype: int64

surface_enc
surface_enc
0     21
1    388
2    245
3      1
Name: count, dtype: int64

light_enc
light_enc
1    152
2    266
3    237
Name: count, dtype: int64

green_bin
green_bin
0      4
1    651
Name: count, dtype: int64


In [ ]:
#compute six emotion scores

In [18]:
# -------------------------------
# Theory-grounded affect formulas
# Scores are clipped to 0–10
# -------------------------------

# STRESS:
# Higher traffic, higher enclosure/building ratio, poorer surfaces,
# lower sky visibility, and lower greenery increase stress.
scores_df["stress"] = np.clip(
    (scores_df["traffic_enc"] / 3 * 3.5)
    + (scores_df["height_enc"] / 3 * 2.5)
    + ((3 - scores_df["surface_enc"]) / 3 * 1.5)
    + ((1 - scores_df["sky_fraction"]) * 1.5)
    + ((3 - scores_df["greenery_enc"]) / 3 * 1.0),
    0, 10
).round(1)

# CALM:
# Greenery, sky visibility, low traffic, and good surface condition increase calm.
scores_df["calm"] = np.clip(
    (scores_df["greenery_enc"] / 3 * 4.0)
    + (scores_df["sky_fraction"] * 3.0)
    + ((3 - scores_df["traffic_enc"]) / 3 * 2.0)
    + (scores_df["surface_enc"] / 3 * 1.0),
    0, 10
).round(1)

# VITALITY:
# Pedestrian activity, good lighting, and good surface condition increase vitality.
scores_df["vitality"] = np.clip(
    (scores_df["pedestrian_enc"] / 3 * 5.0)
    + (scores_df["light_enc"] / 3 * 3.0)
    + (scores_df["surface_enc"] / 3 * 2.0),
    0, 10
).round(1)

# ENCLOSURE:
# High building ratio and low sky visibility increase enclosure.
scores_df["enclosure"] = np.clip(
    (scores_df["height_enc"] / 3 * 6.0)
    + ((1 - scores_df["sky_fraction"]) * 4.0),
    0, 10
).round(1)

# SAFETY:
# Good surface condition, lighting, pedestrian presence, and lower traffic increase perceived safety.
scores_df["safety"] = np.clip(
    (scores_df["surface_enc"] / 3 * 3.5)
    + (scores_df["light_enc"] / 3 * 3.0)
    + (scores_df["pedestrian_enc"] / 3 * 2.0)
    + ((3 - scores_df["traffic_enc"]) / 3 * 1.5),
    0, 10
).round(1)

# DESOLATION:
# Lack of people, poor condition, and low light increase desolation.
scores_df["desolation"] = np.clip(
    ((3 - scores_df["pedestrian_enc"]) / 3 * 4.0)
    + ((3 - scores_df["surface_enc"]) / 3 * 3.0)
    + ((3 - scores_df["light_enc"]) / 3 * 3.0),
    0, 10
).round(1)

score_cols = ["stress", "calm", "vitality", "enclosure", "safety", "desolation"]

scores_df[score_cols].head()

,stress,calm,vitality,enclosure,safety,desolation
0,3.9,6.1,3.7,3.6,5.2,6.0
1,2.7,7.0,6.0,5.6,7.5,3.7
2,4.9,4.5,3.3,3.6,4.8,6.0
3,6.4,3.8,3.3,6.0,3.3,6.7
4,6.4,3.8,2.7,6.0,3.7,7.0


In [19]:
scores_df[score_cols].describe().round(2)

,stress,calm,vitality,enclosure,safety,desolation
count,655.00,655.00,655.00,655.00,655.00,655.00
mean,4.60,5.24,3.57,5.10,4.88,6.10
std,1.38,1.54,1.25,1.69,1.10,1.22
min,1.80,1.60,1.00,3.20,2.00,2.30
25%,3.55,4.15,2.70,3.60,4.20,5.00
50%,4.60,5.20,3.70,4.00,4.80,6.00
75%,5.40,6.20,4.30,6.40,5.70,7.00
max,8.00,8.80,7.70,9.60,8.70,9.00


In [20]:
#distribution check
print("=== Emotion Score Distribution Check ===")

flagged = []

for col in score_cols:
    mean_val = scores_df[col].mean()
    std_val = scores_df[col].std()
    
    flag = ""
    if mean_val < 2.5 or mean_val > 7.5 or std_val < 0.8:
        flag = " <<< FLAG"
        flagged.append(col)
    
    print(f"{col:12s}: mean={mean_val:.2f}, std={std_val:.2f}{flag}")

print("\nFlagged dimensions:", flagged)

=== Emotion Score Distribution Check ===
stress      : mean=4.60, std=1.38
calm        : mean=5.24, std=1.54
vitality    : mean=3.57, std=1.25
enclosure   : mean=5.10, std=1.69
safety      : mean=4.88, std=1.10
desolation  : mean=6.10, std=1.22

Flagged dimensions: []


In [21]:
#rescaling block
APPLY_RESCALING = False

if APPLY_RESCALING and flagged:
    print("Applying min-max rescaling to flagged dimensions:", flagged)
    
    for col in flagged:
        original_col = f"{col}_raw"
        scores_df[original_col] = scores_df[col]
        
        mn = scores_df[col].min()
        mx = scores_df[col].max()
        
        if mx > mn:
            scores_df[col] = ((scores_df[col] - mn) / (mx - mn) * 10).round(1)
            print(f"Rescaled {col}: min={mn:.2f}, max={mx:.2f}")
        else:
            print(f"Skipped {col}: min=max={mn:.2f}")
else:
    print("No rescaling applied.")

No rescaling applied.


In [22]:
#check borough-level summaries
borough_summary = (
    scores_df
    .groupby("borough")[score_cols]
    .mean()
    .round(2)
    .sort_values("stress", ascending=False)
)

borough_summary.head(10)

,stress,calm,vitality,enclosure,safety,desolation
borough,,,,,,
City of London,5.68,3.60,5.12,7.73,5.79,4.77
Hammersmith and Fulham,5.43,4.54,3.58,5.76,4.60,6.16
Tower Hamlets,5.42,4.60,4.10,7.07,5.22,5.65
Kensington and Chelsea,5.36,4.65,4.16,6.40,5.09,5.62
Hackney,5.34,4.86,3.82,6.15,4.73,5.97
Islington,5.24,4.99,3.55,5.98,4.70,6.14
Camden,5.18,4.88,4.00,6.00,5.05,5.75
Waltham Forest,5.04,4.74,3.78,5.08,4.99,5.85
Lambeth,5.02,5.18,3.64,5.89,4.64,6.11


In [23]:
borough_summary.tail(10)

,stress,calm,vitality,enclosure,safety,desolation
borough,,,,,,
Redbridge,4.25,5.48,3.78,3.88,5.32,5.72
Haringey,4.25,5.81,3.92,4.75,5.26,5.77
Hounslow,4.24,5.37,3.26,4.65,4.78,6.36
Harrow,4.05,5.93,3.48,3.80,4.87,6.15
Greenwich,3.90,5.83,3.22,4.24,4.88,6.31
Hillingdon,3.75,6.17,3.42,3.60,5.05,6.14
Croydon,3.73,5.65,2.46,4.61,4.16,7.25
Bromley,3.66,6.33,3.13,4.76,4.98,6.38
Richmond upon Thames,3.66,6.21,3.78,4.24,5.23,5.93


In [24]:
#top/bottom boroughs per emotion
for col in score_cols:
    print("\n" + "="*80)
    print(f"Highest {col}")
    print(borough_summary[col].sort_values(ascending=False).head(5))
    
    print(f"\nLowest {col}")
    print(borough_summary[col].sort_values(ascending=True).head(5))


Highest stress
borough
City of London            5.68
Hammersmith and Fulham    5.43
Tower Hamlets             5.42
Kensington and Chelsea    5.36
Hackney                   5.34
Name: stress, dtype: float64

Lowest stress
borough
Sutton                  3.23
Bromley                 3.66
Richmond upon Thames    3.66
Croydon                 3.73
Hillingdon              3.75
Name: stress, dtype: float64

Highest calm
borough
Bromley                 6.33
Richmond upon Thames    6.21
Hillingdon              6.17
Sutton                  6.14
Harrow                  5.93
Name: calm, dtype: float64

Lowest calm
borough
City of London            3.60
Hammersmith and Fulham    4.54
Barking and Dagenham      4.59
Tower Hamlets             4.60
Kensington and Chelsea    4.65
Name: calm, dtype: float64

Highest vitality
borough
City of London            5.12
Kensington and Chelsea    4.16
Tower Hamlets             4.10
Camden                    4.00
Haringey                  3.92
Name: vitality, d

In [26]:
from scipy.stats import spearmanr

In [27]:
#feature-affect sanity correlations
from scipy.stats import spearmanr

feature_cols = [
    "sky_fraction",
    "greenery_enc",
    "height_enc",
    "traffic_enc",
    "pedestrian_enc",
    "surface_enc",
    "light_enc"
]

correlation_rows = []

for feat in feature_cols:
    for score in score_cols:
        temp = scores_df[[feat, score]].dropna()
        r, p = spearmanr(temp[feat], temp[score])
        
        correlation_rows.append({
            "feature": feat,
            "score": score,
            "spearman_r": r,
            "p_value": p
        })

correlation_df = pd.DataFrame(correlation_rows)
correlation_df.sort_values("spearman_r", ascending=False).head(15)

,feature,score,spearman_r,p_value
15,height_enc,enclosure,0.952596,0.000000e+00
7,greenery_enc,calm,0.899348,8.439853e-237
18,traffic_enc,stress,0.778325,4.052624e-134
40,light_enc,safety,0.712383,1.638905e-102
12,height_enc,stress,0.693935,3.200358e-95
26,pedestrian_enc,vitality,0.668881,3.671531e-86
38,light_enc,vitality,0.668441,5.202050e-86
34,surface_enc,safety,0.602805,4.801678e-66
1,sky_fraction,calm,0.450801,4.211747e-34
32,surface_enc,vitality,0.346945,5.796926e-20


In [28]:
correlation_df.sort_values("spearman_r", ascending=True).head(15)

,feature,score,spearman_r,p_value
3,sky_fraction,enclosure,-0.936522,1.960774e-299
41,light_enc,desolation,-0.640205,8.001387e-77
29,pedestrian_enc,desolation,-0.617906,3.193266e-70
0,sky_fraction,stress,-0.557395,9.752705e-55
13,height_enc,calm,-0.544946,6.224827e-52
6,greenery_enc,stress,-0.527590,3.264430e-48
19,traffic_enc,calm,-0.516818,5.202268e-46
35,surface_enc,desolation,-0.495747,6.354024e-42
9,greenery_enc,enclosure,-0.307004,9.205578e-16
17,height_enc,desolation,-0.212094,4.247861e-08


In [29]:
correlation_df.to_csv(DATA_DIR / "stage3_feature_score_correlations.csv", index=False)

In [30]:
# Clean important ID/path columns before saving
string_cols = [
    "image_id",
    "filepath",
    "borough",
    "road_type",
    "source",
    "scene_description",
    "greenery_density",
    "building_height_ratio",
    "traffic_intensity",
    "pedestrian_activity",
    "surface_condition",
    "lighting"
]

for col in string_cols:
    if col in scores_df.columns:
        scores_df[col] = scores_df[col].astype("string")

# Save Stage 3 outputs
emotion_parquet_path = DATA_DIR / "emotion_scores.parquet"
emotion_csv_path = DATA_DIR / "emotion_scores.csv"

scores_df.to_parquet(emotion_parquet_path, index=False)
scores_df.to_csv(emotion_csv_path, index=False)

print("Saved:", emotion_parquet_path)
print("Saved:", emotion_csv_path)
print("Rows:", len(scores_df))

Saved: /Users/yongyili/urban-emotional-intelligence/data/emotion_scores.parquet
Saved: /Users/yongyili/urban-emotional-intelligence/data/emotion_scores.csv
Rows: 655


In [31]:
check_df = pd.read_parquet(DATA_DIR / "emotion_scores.parquet")

print("Reloaded:", check_df.shape)
check_df[["image_id", "borough"] + score_cols].head()

Reloaded: (655, 33)


,image_id,borough,stress,calm,vitality,enclosure,safety,desolation
0,2567577653535954,Kingston upon Thames,3.9,6.1,3.7,3.6,5.2,6.0
1,227868632440032,Kingston upon Thames,2.7,7.0,6.0,5.6,7.5,3.7
2,1101120203722336,Kingston upon Thames,4.9,4.5,3.3,3.6,4.8,6.0
3,220316326204880,Kingston upon Thames,6.4,3.8,3.3,6.0,3.3,6.7
4,160502959355506,Kingston upon Thames,6.4,3.8,2.7,6.0,3.7,7.0


In [32]:
check_df[score_cols].describe().round(2)

,stress,calm,vitality,enclosure,safety,desolation
count,655.00,655.00,655.00,655.00,655.00,655.00
mean,4.60,5.24,3.57,5.10,4.88,6.10
std,1.38,1.54,1.25,1.69,1.10,1.22
min,1.80,1.60,1.00,3.20,2.00,2.30
25%,3.55,4.15,2.70,3.60,4.20,5.00
50%,4.60,5.20,3.70,4.00,4.80,6.00
75%,5.40,6.20,4.30,6.40,5.70,7.00
max,8.00,8.80,7.70,9.60,8.70,9.00
